# Laya post-training on Colab — curated datasets + SFT & RLCD reinforcement learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tshewangrinzin/laya/blob/arena/01a0c378-laya/notebooks/laya_colab_post_training_rl.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

Fine-tune the **Laya English checkpoint** (`convaiinnovations/laya`, ModernBERT-large 421 M) for
precision and calibration, entirely in this notebook:

1. **Clone the repo** and run its `datasets/curate.py` — a from-scratch curation pipeline (fetch
   from **Hugging Face + Kaggle** → clean → filter → label-normalise → dedupe → quality-score →
   balance → leak-safe splits) that produces the JSONL datasets in `datasets/curated/`
2. **Pre-tokenise** every case into Laya's `[CLS] question [SEP] [MASK] opt0 … state [SEP]` layout
3. **Post-train** in two stages: **SFT** on the teacher distributions, then **RLCD reinforcement
   learning** — GRPO-style policy gradient where the reward is a *strictly proper scoring rule*
   (log + spherical + ranked-probability score), so the only way to maximise reward is to report
   honest probabilities — periodic validation + best-checkpoint selection, then temperature
   calibration fitted on a held-out *calibration* slice
4. **Evaluate** on a clean test split: accuracy, Brier, ECE, score-MAE, base-vs-fine-tuned table
5. **Push** the result to the Hub — then [laya_colab_openai_api.ipynb](https://github.com/tshewangrinzin/laya/blob/arena/01a0c378-laya/notebooks/laya_colab_openai_api.ipynb)
   serves it as `laya-ft` behind the OpenAI-compatible API + Cloudflare tunnel

This is the Colab sibling of the repo's
[`notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb`](https://github.com/tshewangrinzin/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)
(same RLCD recipe, single-GPU by default, curated-mix data instead of benchmark-only, adds SFT
warmup + by-options temperature buckets). Reference scale: the typed-decisions benchmark goes
**0.362 → 0.766** accuracy with this family of methods while ECE improves with temperature
fitting — post-training is where most of Laya's value is.

**Runtime:** Edit → Notebook settings → **T4 GPU** (free tier suffices: gradient checkpointing +
fp16 + grad-accum keep peak VRAM ≈ 8–11 GB). Defaults curate a ~15–25 k-case mix and train 4 epochs
in roughly 1.5–3 h on one T4; `SMOKE=True` runs the whole loop in ~10 min to prove the plumbing.

## 1 · Configuration

In [ ]:
# ------------------------------------------------------------------ CONFIG
REPO_URL  = "https://github.com/tshewangrinzin/laya.git"
REPO_DIR  = "/content/laya"
DATA_DIR  = "/content/laya_data"
OUT_DIR   = "/content/laya_ft"

MODEL = "laya"                       # base checkpoint to start from (english)
CURATE_SOURCES = "all"               # "all" | "typed_decisions,typed_decisions_synth,..." | "demo"
CURATE_KAGGLE  = False               # True -> also pull the 2 optional Kaggle sources (needs creds)
CURATE_LIMIT   = 6000                # max raw rows per source/split (curated mix ~= 15-25k cases)
SMOKE = False                        # tiny data + tiny epochs: ~10 min end-to-end plumbing test

# hyper-parameters (defaults tuned for one T4; see the reference 2xT4 recipe for provenance)
EPOCHS = 4; WARMUP_EPOCHS = 1         # epoch 1 = SFT, then RLCD
MICRO_BATCH = 4; GRAD_ACCUM = 8        # effective batch 32 sequences
GROUP_SIZE = 4                         # GRPO samples per question (group-mean baseline)
LR_ENCODER = 2.5e-5; LR_HEAD = 1.0e-4
SIGMA_START = 0.4; SIGMA_END = 0.1     # exploration noise schedule (logit space)
CE_W = 1.0                             # weight of teacher soft-CE alongside the RL loss
SEED = 42

HF_PUSH = False                        # True -> push final checkpoint to the Hub
HF_REPO = "laya-ft-colab"              # will be created under your account


In [ ]:
import os, sys, platform

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only || true
os.makedirs(DATA_DIR, exist_ok=True)

%pip install -q -U "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub     numpy pandas pyarrow matplotlib scipy
!pip install -q -e {REPO_DIR} 2>/dev/null || pip install -q laya
if CURATE_KAGGLE:
    !pip install -q kaggle kagglehub

import laya, datasets, transformers
print("laya", laya.__version__, "| datasets", datasets.__version__, "| transformers", transformers.__version__)

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU (CPU training is ~15x slower)"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
print("gpus:", torch.cuda.device_count())

## 2 · (Optional) credentials

* **Kaggle** (only for `CURATE_KAGGLE=True`): upload your `kaggle.json` (Account → Create New Token) —
  the cell writes it to `~/.kaggle/`.
* **Hugging Face** (only for `HF_PUSH=True` or a gated base model): a *write* token from
  <https://huggingface.co/settings/tokens>.

In [ ]:
# Kaggle credentials (skip unless CURATE_KAGGLE)
if CURATE_KAGGLE:
    from google.colab import files
    import shutil
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("Upload kaggle.json now ...")
        uploaded = files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        for name, blob in uploaded.items():
            open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb").write(blob)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    !kaggle datasets list -s "customer support" -f "customer support ticket" 2>/dev/null | head -3 || true
    print("kaggle creds ready")

# HF token for pushing
if HF_PUSH:
    from huggingface_hub import notebook_login
    notebook_login()

## 3 · Curate the datasets

`datasets/curate.py` (from the repo you just cloned) fetches raw rows from **open Hugging Face
datasets — and optionally Kaggle —** and applies the full processing chain, in order:

| stage | what happens |
|---|---|
| fetch | `LocalLLaMA/typed-decisions` (the RLCD benchmark), `n4ze3m/typed-decisions-synth` (MIT, teacher soft labels), `fancyzhx/ag_news`, `dair-ai/emotion`, `clinc/clinc_oos`, `ucirvine/sms_spam`, `deepset/prompt-injections`, `SetFit/sst5`; Kaggle: `waseemalastal/customer-support-ticket-dataset`, `scodepy/customer-support-intent-dataset` |
| catalog | choice catalogs rebuilt dataset-wide; long tails (CLINC's 150 intents…) folded to `other` so options stay legible inside `head_max_len=192` |
| clean | mojibake repair (UTF-8-as-latin-1 round-trip), HTML/entity stripping, unicode NFC, whitespace collapse, signature removal, **PII scrubbing** (URLs/e-mails/long account numbers → placeholders) |
| filter | English gate (script + stopword coverage), length bands, garbage/digit-ratio/repetition rejects |
| label-norm | canonical label slugs, soft targets clipped from 0/1 + label smoothing (no raw one-hots — proper scoring wants honest targets) |
| dedupe | exact-hash **and** MinHash/LSH near-duplicate removal over state text (cross-source too) |
| quality | per-case 0-1 score (length/diversity/margin/cleanliness); low-quality drops; low teacher-margin → routed to `calib` |
| split | 76/8/16 parts-per-1000 of the clean pool **by case** (a state never appears in two splits) + official test rows forced into `test` |
| balance | per-(question, catalog) label caps on **train only**; val/test keep the natural mix |

It writes `datasets/curated/laya_posttrain_{train,val,calib,test}.jsonl` + `curated_report.json`
with per-stage drop counts — if any source is unreachable it's skipped and reported, never fatal
(`--demo` runs the same pipeline on built-in dirty synthetic rows so the notebook never stalls).

In [ ]:
import json, subprocess

CUR_OUT = os.path.join(REPO_DIR, "datasets", "curated")
cur_cmd = [sys.executable, os.path.join(REPO_DIR, "datasets", "curate.py"),
           "--sources", CURATE_SOURCES, "--out-dir", CUR_OUT,
           "--per-source-limit", str(CURATE_LIMIT), "--seed", str(SEED)]
if CURATE_KAGGLE:
    cur_cmd.append("--with-kaggle")
if SMOKE:
    cur_cmd += ["--per-source-limit", "300", "--min-quality", "0.1", "--balance-min", "10"]

rc = subprocess.run(cur_cmd).returncode
if rc != 0:
    print("curate.py exited %d -> falling back to the offline --demo source so the rest still runs" % rc)
    subprocess.run([sys.executable, os.path.join(REPO_DIR, "datasets", "curate.py"), "--demo",
                    "--out-dir", CUR_OUT], check=True)

report = json.load(open(os.path.join(CUR_OUT, "curated_report.json")))
print("\ncounts   :", report["counts"])
print("primitives:", report["primitive_counts"])
print("stats    :", json.dumps(report["stats"], indent=1))

## 4 · Dataset QC

Quality-gate checks before spending GPU-hours: schema validity, probability normalisation, split
leakage, near-duplicate survival, label coverage. **Every assertion must pass** — they are the
contract the trainer relies on.

In [ ]:
import hashlib, json, random
import pandas as pd

splits = {}
for name in ("train", "val", "calib", "test"):
    path = os.path.join(CUR_OUT, "laya_posttrain_%s.jsonl" % name)
    if os.path.exists(path):
        splits[name] = [json.loads(l) for l in open(path)]
print({k: len(v) for k, v in splits.items()})

def _norm(s):
    import re as _re
    return _re.sub(r"\W+", "", s.lower())

seen_state, seen_id = set(), set()
for name, rows in splits.items():
    for r in rows:
        q = json.loads(r["questions"]); g = json.loads(r["gold"])
        assert r["id"] not in seen_id, "duplicate id"
        seen_id.add(r["id"])
        seen_state.add((name, _norm(r["state"])))
        assert set(q) >= set(g) and q, (name, r["id"])
        for qid, gg in g.items():
            p = gg["probabilities"]
            assert abs(sum(p.values()) - 1) < 2e-3, (r["id"], qid)
            assert all(0.005 <= v <= 0.995 for v in p.values()), "unclipped prob"
            if q[qid]["type"] == "choice":
                assert gg["label"] in q[qid]["criteria"] and len(q[qid]["criteria"]) >= 2
            if q[qid]["type"] == "score":
                assert gg["label"] in [str(i) for i in range(len(q[qid]["criteria"]))]
# cross-split state leakage
by = {}
for (name, key) in seen_state:
    by.setdefault(key, set()).add(name)
overlap = [k for k, v in by.items() if len(v) > 1]
assert not overlap, "state appears in multiple splits: %s" % overlap[:3]

df = pd.DataFrame([{"split": n, "workflow": r["workflow"], "source": r["source"],
                    "quality": r["quality"], "nq": r["n_questions"]} for n, rows in splits.items() for r in rows])
print(df.groupby("split").agg(n=("quality", "size"), mean_quality=("quality", "mean")).head(20))
print("\nsources:", df["source"].value_counts().to_dict())
print("workflows:", df["workflow"].value_counts().head(20).to_dict())

# eyeball a random sample from each primitive
show = []
for name, rows in list(splits.items())[:1]:
    for r in random.Random(SEED).sample(rows, min(4, len(rows))):
        show.append({"id": r["id"], "state": r["state"][:160],
                     "gold": {k: json.loads(r["gold"])[k] for k in list(json.loads(r["gold"]))[:2]}})
for s in show:
    print("\n" + json.dumps(s, indent=1)[:520])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df[df.split == "train"]["quality"].plot.hist(bins=25, ax=axes[0], color="#3b7")
axes[0].set_title("train quality scores")
df["workflow"].value_counts().head(12).plot.barh(ax=axes[1], color="#59c")
axes[1].set_title("workflows"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 5 · Pre-tokenise into Laya decision sequences

Every case becomes `(ids, marker positions, qtype, soft target, label)` under the English
checkpoint's `max_len=512 / head_max_len=192` budget — **exactly** the tensor format the model was
trained on (`laya.common.build_sequence`). Rows whose options don't fit the head budget are dropped
here (curation's option folding should have kept this at zero).

In [ ]:
import json, os
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

SUBFOLDERS = {"laya": None, "laya-multilingual": "multilingual", "laya-typed-decisions": "typed-decisions"}
model_id = "convaiinnovations/laya" if MODEL == "laya" else "convaiinnovations/" + MODEL
model_dir = snapshot_download(model_id)
_fix_tokenizer_config(model_dir)
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
MAX_LEN, HEAD_MAX_LEN = cfg.get("max_len", 512), cfg.get("head_max_len", 192)
print("tokenising with max_len=%d head_max_len=%d" % (MAX_LEN, HEAD_MAX_LEN))

def build_items(rows):
    items, dropped = [], 0
    for r in rows:
        state = json.loads(r["state"])
        questions = json.loads(r["questions"])
        gold = json.loads(r["gold"])
        for qid, q in questions.items():
            if qid not in gold:
                dropped += 1; continue
            t, crit = q["type"], q.get("criteria", {} if q["type"] != "score" else [])
            g = gold[qid]
            opts = list(crit.keys()) if t == "choice" else [str(i) for i in range(len(crit))] if t == "score" else ["false", "true"]
            target = [float(g["probabilities"].get(k, 0.0)) for k in opts]
            s = sum(target)
            target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
            qin = {"t": t, "ins": q["instructions"], "crit": crit}
            seq, markers = build_sequence(tok, state, qin, MAX_LEN, HEAD_MAX_LEN)
            if len(markers) != len(render_options(qin)):
                dropped += 1; continue
            items.append({"ids": seq, "markers": markers, "qtype": QTYPES[t],
                          "target": target, "label": target.index(max(target))})
    return items, dropped

for name in list(splits):
    items, dropped = build_items(splits[name])
    if name == "train" and SMOKE:
        items = items[:256]
    torch.save(items, os.path.join(DATA_DIR, "%s_items.pt" % name))
    print("%-6s %7d tensors | dropped %d" % (name, len(items), dropped))
_tr = torch.load(os.path.join(DATA_DIR, "train_items.pt"), weights_only=False)
lens = [len(it["ids"]) for it in _tr[:2000]]
print("token length: mean %.0f p99 %.0f max %d" % (sum(lens)/len(lens), sorted(lens)[int(.99*len(lens))], max(lens)))

# the eval cells also need the raw test rows next to the tensors
json.dump(splits["test"], open(os.path.join(DATA_DIR, "test_cases.json"), "w"))

### 5.1 · Back up data + tensors to Drive (recommended)

In [ ]:
# keep a copy of the curated JSONL + report alongside the run outputs (Colab disk is ephemeral)
import shutil
for fn in sorted(os.listdir(CUR_OUT)):
    shutil.copy(os.path.join(CUR_OUT, fn), os.path.join(DATA_DIR, fn))
try:
    from google.colab import drive
    drive.mount("/content/drive")
    out = "/content/drive/MyDrive/laya_data_%s" % os.environ.get("USER", "colab")
    os.makedirs(out, exist_ok=True)
    for fn in sorted(os.listdir(DATA_DIR)):
        shutil.copy(os.path.join(DATA_DIR, fn), out)
    print("backed up curated data + tensors to Drive:", out)
except Exception as e:
    print("(Drive backup skipped:", e, ")")

## 6 · Post-training: SFT warmup → RLCD reinforcement learning

The trainer (written to `/content/laya_colab_train.py` below) implements the RLCD objective from
the repo, single-GPU tuned:

```
for each mini-batch:
  z0 = forward(state, options)                          # logits at the [MASK] option markers
  if epoch < WARMUP_EPOCHS:  loss = softCE(z0, teacher) # SFT phase
  else:                                                 # RLCD phase
    z_g   = z0.detach() + zero-mean gaussian noise(GROUP_SIZE)   # exploration
    q_g   = softmax(z_g)                                        # the "policy" reports distributions
    r_g   = log(q_g·t) + 0.75·cos(q_g, t) − 1.0·RPS(q_g, t)     # STRICTLY PROPER scoring reward
    adv   = (r_g − mean(r_g)) / std(r_g)                        # group-relative baseline (GRPO)
    loss  = −mean(adv · gaussian_logprob(z_g | z0, sigma)) + CE_W · softCE(z0, t)
```

Because the reward is a *strictly proper scoring rule*, expected reward is maximised **only** by
honest probabilities — accuracy and calibration improve together (that's the whole RLCD idea;
`laya.common.proper_reward` is reused verbatim). Validation runs every epoch (acc / Brier / ECE on
the val slice) and the best checkpoint wins; afterwards a temperature is fitted per question type
**and** per option-count bucket on the `calib` slice and written into `rl_agent_config.json`, so
`laya.load(dir)` and the API server serve it calibrated. Peak VRAM ≈ 8–11 GB with gradient
checkpointing + fp16, i.e. one free T4.

If the Colab session dies mid-training: the curated JSONL + `*_items.pt` tensors are backed up to Drive by the previous cell — re-run §1–§5, restore `/content/laya_data` from the Drive folder, and re-run this cell (the trainer is seeded, so it reproduces).

In [ ]:
%%writefile /content/laya_colab_train.py
"""Laya post-training for Colab: SFT warmup on teacher distributions, then RLCD reinforcement
training with strictly proper scoring-rule rewards (GRPO-style policy gradient), periodic
validation, and post-hoc temperature calibration.

Written to /content/laya_colab_train.py by the notebook and run as:

    python /content/laya_colab_train.py /content/laya_model /content/laya_ft

  * items were preprocessed by the notebook into /content/laya_data/{train,val,calib}_items.pt
    (lists of dicts: ids, markers, qtype, target, label) -- the exact format laya's collate uses
  * reward = log score + w_sph * spherical score ( - w_rps * ranked probability score for the
    ordinal `score` primitive): strictly proper, so the policy maximises expected reward only
    by reporting honest probabilities (this is what RLCD means here)
  * exploration = zero-mean Gaussian noise on the logits; advantage = reward minus group mean,
    standardised within the group (GRPO baseline, no learned critic)
  * single process on one GPU; if launched under torchrun with 2 GPUs it uses DDP identically
  * final artifacts in output dir: model.safetensors (fp16) + encoder/ + tokenizer/ +
    rl_agent_config.json (with fitted temperatures) -- i.e. a directory laya.load() accepts

Environment knobs (set by the notebook): LR_ENCODER, LR_HEAD, EPOCHS, WARMUP_EPOCHS,
MICRO_BATCH, GRAD_ACCUM, GROUP_SIZE, SIGMA_START, SIGMA_END, CE_W, VAL_EVAL_ITEMS, SEED.
"""
import json
import os
import random
import sys
import time

import numpy as np
import torch
from safetensors.torch import load_file, save_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

from laya.common import QTYPES, build_model, ece_score, proper_reward

OUT = {}


def env(k, cast, dflt):
    v = os.environ.get(k, "")
    try:
        return cast(v) if v != "" else dflt
    except ValueError:
        return dflt


def collate_train_batch(items, pad_id):
    n = len(items)
    L = max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
            "label": torch.tensor([it["label"] for it in items])}


def fit_temperatures(preds):
    """One temperature per qtype + one per (qtype, #options bucket); returns (list3, dict)."""
    def _fit(sel):
        if len(sel) < 10:
            return 1.0
        kmax = max(len(z) for z, _ in sel)
        Z = torch.full((len(sel), kmax), -1e4)
        T = torch.zeros((len(sel), kmax))
        for i, (z, t) in enumerate(sel):
            Z[i, :len(z)] = torch.tensor(z)
            T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
        log_t = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

        def closure():
            opt.zero_grad()
            loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
            loss.backward()
            return loss
        opt.step(closure)
        return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

    def bucket(k):
        return "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"

    names = {0: "choice", 1: "score", 2: "noul"}
    temps = [1.2, 1.2, 1.2]
    by_options = {}
    for qt in range(3):
        sel = [(z, t) for q, z, t in preds if q == qt]
        if sel:
            temps[qt] = _fit(sel)
        for bk in ("2", "3-5", "6-10", "11+"):
            s2 = [(z, t) for q, z, t in preds if q == qt and bucket(len(z)) == bk]
            if len(s2) >= 25:
                by_options["%s:%s" % (names[qt], bk)] = _fit(s2)
    return temps, by_options


@torch.no_grad()
def evaluate(model, device, items, pad_id, tok, max_items=2000, bs=32):
    model.eval()
    items = items[:max_items]
    accs, briers, confs, correct = [], [], [], []
    for b0 in range(0, len(items), bs):
        batch = collate_train_batch(items[b0:b0 + bs], pad_id)
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                              batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                              batch["qtype"].to(device))
        mask = batch["marker_mask"].to(device)
        logits = logits.float().masked_fill(~mask, -1e4)
        p = torch.softmax(logits, -1)
        t = batch["target"].to(device)
        pred = p.argmax(-1).cpu().numpy()
        label = batch["label"].cpu().numpy()
        accs.append((pred == label).mean())
        briers.append(((p - t) ** 2).sum(-1).mean().item())
        top_p, _ = p.max(-1)
        confs.append(top_p.cpu().numpy())
        correct.append((pred == label).astype(float))
    model.train()
    return {"val_acc": round(float(np.mean(accs)), 4), "val_brier": round(float(np.mean(briers)), 4),
            "val_ece": round(ece_score(np.concatenate(confs), np.concatenate(correct)), 4)}


def main():
    rank = int(os.environ.get("RANK", "0"))
    world = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    ddp = world > 1 and torch.cuda.is_available()
    if ddp:
        import torch.distributed as dist
        dist.init_process_group("nccl")
        torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")
    is_main = rank == 0

    model_dir, output_dir = sys.argv[1], sys.argv[2]
    data_dir = os.environ.get("LAYA_DATA_DIR", "/content/laya_data")
    hp = dict(EPOCHS=env("EPOCHS", int, 4), WARMUP_EPOCHS=env("WARMUP_EPOCHS", int, 1),
              MICRO_BATCH=env("MICRO_BATCH", int, 4), GRAD_ACCUM=env("GRAD_ACCUM", int, 8),
              GROUP_SIZE=env("GROUP_SIZE", int, 4), LR_ENCODER=env("LR_ENCODER", float, 2.5e-5),
              LR_HEAD=env("LR_HEAD", float, 1e-4), SIGMA_START=env("SIGMA_START", float, 0.4),
              SIGMA_END=env("SIGMA_END", float, 0.1), CE_W=env("CE_W", float, 1.0),
              VAL_EVAL_ITEMS=env("VAL_EVAL_ITEMS", int, 600), SEED=env("SEED", int, 42))
    torch.manual_seed(hp["SEED"] + rank)
    random.seed(hp["SEED"] + rank)
    np.random.seed(hp["SEED"] + rank)

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    try:
        model.encoder.config.reference_compile = False
    except Exception:
        pass
    if torch.cuda.is_available():
        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.head_checkpointing = True
    model.to(device)
    model.train()
    if ddp:
        model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    raw_model = model.module if ddp else model

    train_items = torch.load(os.path.join(data_dir, "train_items.pt"), weights_only=False)
    val_items = []
    if os.path.exists(os.path.join(data_dir, "val_items.pt")):
        val_items = torch.load(os.path.join(data_dir, "val_items.pt"), weights_only=False)
    calib_items = []
    if os.path.exists(os.path.join(data_dir, "calib_items.pt")):
        calib_items = torch.load(os.path.join(data_dir, "calib_items.pt"), weights_only=False)
    if ddp:
        train_items = train_items[rank::world]
        val_items = val_items[rank::world] if val_items else []
        calib_items = calib_items[rank::world] if calib_items else []
    if is_main:
        print("[train] items: train=%d val=%d calib=%d | hp=%s"
              % (len(train_items), len(val_items), len(calib_items), hp), flush=True)

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": hp["LR_ENCODER"]},
                                   {"params": head_params, "lr": hp["LR_HEAD"]}], weight_decay=0.01)
    steps_per_epoch = max(1, len(train_items) // (hp["MICRO_BATCH"] * hp["GRAD_ACCUM"]))
    total_updates = steps_per_epoch * hp["EPOCHS"]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    best = {"val_acc": -1.0}
    history = []
    t0 = time.time()
    for epoch in range(hp["EPOCHS"]):
        phase = "sft" if epoch < hp["WARMUP_EPOCHS"] else "rlcd"
        random.shuffle(train_items)
        optimizer.zero_grad(set_to_none=True)
        accum, n_b, loss_sum, r_sum = 0, 0, 0.0, 0.0
        progress = epoch / max(1, hp["EPOCHS"] - 1)
        sigma = hp["SIGMA_START"] + (hp["SIGMA_END"] - hp["SIGMA_START"]) * progress
        for b0 in range(0, len(train_items), hp["MICRO_BATCH"]):
            chunk = train_items[b0:b0 + hp["MICRO_BATCH"]]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, pad_id)
            with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits, act = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                                    batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                                    batch["qtype"].to(device))
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            if phase == "sft":
                loss = loss_ce / hp["GRAD_ACCUM"] + 0.0 * act.sum()
                r_mean = float("nan")
            else:
                eps = torch.randn((hp["GROUP_SIZE"],) + logits.shape, device=device) * sigma * mask
                eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
                z = logits.detach().unsqueeze(0) + eps
                q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
                with torch.no_grad():
                    r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask,
                                      w_sph=0.75, w_rps=1.0)
                    adv = r - r.mean(0, keepdim=True)
                    adv = adv / (adv.std() + 1e-6)
                logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
                loss_rl = -(adv * logp).mean()
                loss = (loss_rl + hp["CE_W"] * loss_ce) / hp["GRAD_ACCUM"] + 0.0 * act.sum()
                r_mean = r.mean().item()
            scaler.scale(loss).backward()
            accum += 1
            if accum % hp["GRAD_ACCUM"] == 0 or (b0 + hp["MICRO_BATCH"]) >= len(train_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            loss_sum += loss.item() * hp["GRAD_ACCUM"]
            r_sum += 0.0 if np.isnan(r_mean) else r_mean
            n_b += 1
            if is_main and n_b % 40 == 0:
                print("[train] ep%d/%s step %d | loss %.4f | reward %.3f | lr %.2e | %.0fs"
                      % (epoch + 1, phase, n_b, loss_sum / n_b, r_sum / max(1, n_b) if phase == "rlcd" else float("nan"),
                         scheduler.get_last_lr()[0], time.time() - t0), flush=True)
        m = {"epoch": epoch + 1, "phase": phase, "avg_loss": round(loss_sum / max(1, n_b), 4)}
        m.update(evaluate(raw_model, device, val_items, pad_id, tok, hp["VAL_EVAL_ITEMS"]) if val_items else {})
        history.append(m)
        if is_main:
            print("[eval ] ep%d %s" % (epoch + 1, json.dumps(m)), flush=True)
        if val_items and m.get("val_acc", -1) > best["val_acc"]:
            best = {"val_acc": m["val_acc"], "epoch": epoch + 1}
            if is_main:
                os.makedirs(output_dir, exist_ok=True)
                sd = {kk: v.detach().half().contiguous().cpu() for kk, v in raw_model.state_dict().items()}
                save_file(sd, os.path.join(output_dir, "model.safetensors"))
                with open(os.path.join(output_dir, "best_metrics.json"), "w") as f:
                    json.dump(m, f, indent=2)

    # ---- calibration + final save (rank 0 only under DDP) --------------------------------
    if is_main:
        raw_model.eval()
        preds = []
        calib_items = calib_items or train_items
        if calib_items:
            with torch.no_grad():
                for b0 in range(0, len(calib_items), 32):
                    c = calib_items[b0:b0 + 32]
                    cb = collate_train_batch(c, pad_id)
                    with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                        l, _ = raw_model(cb["input_ids"].to(device), cb["attention_mask"].to(device),
                                          cb["marker_pos"].to(device), cb["marker_mask"].to(device),
                                          cb["qtype"].to(device))
                    l = l.float().masked_fill(~cb["marker_mask"].to(device), -1e4)
                    ln = l.cpu().numpy()
                    for i, it in enumerate(c):
                        kk = len(it["markers"])
                        preds.append((it["qtype"], ln[i, :kk], it["target"]))
            temps, by_options = fit_temperatures(preds)
            if len(preds) >= 30:
                print("[calib] temperatures per qtype:", [round(t, 3) for t in temps],
                      "| by options:", {kk: round(v, 3) for kk, v in by_options.items()}, flush=True)
        else:
            temps, by_options = [1.0, 1.0, 1.0], {}
        os.makedirs(output_dir, exist_ok=True)
        sd = {kk: v.detach().half().contiguous().cpu() for kk, v in raw_model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        raw_model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["temperature"] = temps
        cfg["temperature_by_options"] = by_options
        cfg["finetune_report"] = {"history": history, "best": best, "seconds": round(time.time() - t0, 1),
                                  "hp": hp, "n_train_items": len(train_items)}
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        with open(os.path.join(output_dir, "train_history.json"), "w") as f:
            json.dump({"history": history, "best": best}, f, indent=2)
        print("[done ] saved %s (%.1f min)" % (output_dir, (time.time() - t0) / 60), flush=True)
    if ddp:
        import torch.distributed as dist
        dist.barrier()
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys, os, time

hp_env = {**os.environ,
          "EPOCHS": str(1 if SMOKE else EPOCHS), "WARMUP_EPOCHS": str(WARMUP_EPOCHS),
          "MICRO_BATCH": str(MICRO_BATCH), "GRAD_ACCUM": str(GRAD_ACCUM), "GROUP_SIZE": str(GROUP_SIZE),
          "LR_ENCODER": str(LR_ENCODER), "LR_HEAD": str(LR_HEAD),
          "SIGMA_START": str(SIGMA_START), "SIGMA_END": str(SIGMA_END), "CE_W": str(CE_W),
          "SEED": str(SEED), "LAYA_DATA_DIR": DATA_DIR,
          "VAL_EVAL_ITEMS": "120" if SMOKE else "600"}

t0 = time.time()
# stream the trainer's stdout live (logs also persist to file)
with open("/content/train.log", "a") as logf:
    proc = subprocess.Popen([sys.executable, "/content/laya_colab_train.py", model_dir, OUT_DIR],
                            env=hp_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True); logf.write(line)
    rc = proc.wait()
assert rc == 0, "trainer exited %d -- see /content/train.log" % rc
print("\ntraining finished in %.1f min; checkpoint at %s" % ((time.time() - t0) / 60, OUT_DIR))
import glob
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    print("  %10d KB  %s" % (os.path.getsize(f) // 1024, os.path.basename(f)) if os.path.isfile(f)
          else "     <dir>     %s" % os.path.basename(f))

In [ ]:
import json, os
hist_path = os.path.join(OUT_DIR, "train_history.json")
if os.path.exists(hist_path):
    h = json.load(open(hist_path))
    print("best:", h.get("best"))
    import pandas as pd
    print(pd.DataFrame(h["history"]).to_string(index=False))

## 7 · Evaluate: base vs fine-tuned (accuracy *and* calibration)

The same clean question packs as curation's `test` split are asked to both checkpoints through the
*official* `laya.Agent` path — the fine-tuned one automatically applies its fitted temperatures. We
report what the README reports: **accuracy** (argmax vs teacher label), **top-agreement with the
teacher's distribution**, **Brier** and **ECE** (lower = better calibrated), `score`-MAE, and
latency. ECE improvement is the calibration half of the RLCD win.

In [ ]:
import json, time
import numpy as np
import laya
from laya.common import ece_score

test_cases = json.load(open(os.path.join(DATA_DIR, "test_cases.json")))
if SMOKE:
    test_cases = test_cases[:60]

def evaluate(agent, cases, bs_note=""):
    acc_s, agree_s, briers, confs, correct, maes = [], [], [], [], [], []
    lat = []
    for r in cases:
        state = json.loads(r["state"]); questions = json.loads(r["questions"]); gold = json.loads(r["gold"])
        t0 = time.perf_counter()
        out = agent.predict(state, questions)
        lat.append((time.perf_counter() - t0) * 1e3)
        for qid, g in gold.items():
            a = out["answers"][qid]
            gp = g["probabilities"]
            if g["type"] == "choice":
                mp = a["probabilities"]; pred = a["choice"]
            elif g["type"] == "score":
                mp = a["probabilities"]; pred = str(max(mp, key=mp.get))
            else:
                p = a["noul"]; mp = {"false": 1 - p, "true": p}; pred = a["noul"] >= 0.5
            gl = str(g["label"])
            if g["type"] == "noul":
                acc_s.append(float((pred) == (gl == "true")))
                agree_s.append(float((max(mp, key=mp.get) == "true") == (gl == "true")))
                briers.append((mp["true"] - float(gp.get("true", 0.5))) ** 2)
                confs.append(max(mp["true"], 1 - mp["true"]))
                correct.append(float((pred) == (gl == "true")))
            else:
                if isinstance(gp, dict) and set(map(str, gp)) == set(map(str, mp)):
                    briers.append(sum((float(mp[k]) - float(gp[k])) ** 2 for k in gp) / max(2, len(gp)))
                pred_s = pred if g["type"] == "choice" else str(int(pred)) if isinstance(pred, (int, float)) else str(pred)
                acc_s.append(float(pred_s == gl))
                agree_s.append(float(max(mp, key=mp.get) == max(gp, key=gp.get)))
                confs.append(a["confidence"])
                correct.append(float(pred_s == gl))
                if g["type"] == "score":
                    maes.append(abs(float(a["score"]) - sum(float(i) * float(v) for i, v in gp.items())))
    return {"n_cases": len(cases), "n_decisions": len(acc_s),
            "accuracy": round(float(np.mean(acc_s)), 3),
            "teacher_agreement": round(float(np.mean(agree_s)), 3),
            "brier": round(float(np.mean(briers)), 4),
            "ece": round(ece_score(np.array(confs), np.array(correct)), 4),
            "score_mae": round(float(np.mean(maes)), 3) if maes else None,
            "p50_ms": round(float(np.median(lat)), 1)}

SUBFOLDERS = {"laya": None, "laya-multilingual": "multilingual", "laya-typed-decisions": "typed-decisions"}
mid = "convaiinnovations/laya" if MODEL == "laya" else "convaiinnovations/" + MODEL
base_agent = laya.load(mid, subfolder=SUBFOLDERS[MODEL])
ft_agent = laya.load(OUT_DIR)

import pandas as pd
res = {"base": evaluate(base_agent, test_cases), "fine-tuned": evaluate(ft_agent, test_cases)}
print(pd.DataFrame(res).T.to_string())
json.dump(res, open("/content/eval_report.json", "w"), indent=2)
if res["fine-tuned"]["accuracy"] <= res["base"]["accuracy"]:
    print("\nNOTE: no accuracy gain from this run (SMOKE tiny-training can do that). "
          "Raise EPOCHS/data (SMOKE=False) or use more sources.")

In [ ]:
# per-workflow breakdown of the fine-tuned model
import pandas as pd, json
rows = []
for name, agent in (("base", base_agent), ("fine-tuned", ft_agent)):
    from collections import defaultdict
    acc = defaultdict(lambda: [0, 0])
    for r in test_cases:
        questions = json.loads(r["questions"]); gold = json.loads(r["gold"])
        out = agent.predict(json.loads(r["state"]), questions)
        for qid, g in gold.items():
            a = out["answers"][qid]
            if g["type"] == "noul":
                ok = (a["noul"] >= 0.5) == (g["label"] == "true")
            else:
                pred = a["choice"] if g["type"] == "choice" else str(max(a["probabilities"], key=a["probabilities"].get))
                ok = pred == str(g["label"])
            acc[r["workflow"]][0] += int(ok); acc[r["workflow"]][1] += 1
    for wf, (c, n) in acc.items():
        rows.append({"model": name, "workflow": wf, "acc": round(c / max(1, n), 3), "n": n})
print(pd.DataFrame(rows).pivot_table(index="workflow", columns="model", values="acc").to_string())

## 8 · Publish & serve the improved model

`HF_PUSH=True` uploaded it to the Hub? Then on any machine — including the API notebook — set
`LAYA_FT_REPO="<your-username>/laya-ft-colab"` and it appears as `laya-ft` on every endpoint. In the
same session, `/content/laya_ft` is already picked up automatically (the serve notebook sets
`LAYA_FT_PATH` when the directory exists).

In [ ]:
if HF_PUSH:
    from huggingface_hub import HfApi
    api = HfApi()
    who = api.whoami()
    repo = who["name"] + "/" + HF_REPO
    try:
        api.create_repo(repo, exist_ok=True)
    except Exception as e:
        print("create_repo:", e)
    card = f"""---
license: apache-2.0
base_model: convaiinnovations/laya
tags: [rlcd, decision-model, calibration, laya]
---

# {HF_REPO}

Laya English checkpoint post-trained with SFT + RLCD from `laya_colab_post_training_rl.ipynb`
(curated open-source mix, see `datasets/` in tshewangrinzin/laya). Fitted per-qtype and per-option-count
temperatures are embedded in `rl_agent_config.json`.
"""
    open(os.path.join(OUT_DIR, "README.md"), "w").write(card)
    api.upload_folder(folder_path=OUT_DIR, repo_id=repo, repo_type="model")
    print("pushed:", "https://huggingface.co/" + repo)
else:
    print("HF_PUSH is False -- skipping. Turn it on to publish, or zip below and keep the folder.")
    if os.path.isdir(OUT_DIR):
        !du -sh {OUT_DIR}

In [ ]:
# keep the checkpoint out of the ephemeral graveyard
import shutil, os
if os.path.isdir(OUT_DIR):
    !cd /content && zip -qr laya_ft.zip laya_ft && du -h laya_ft.zip
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copy("/content/laya_ft.zip", "/content/drive/MyDrive/laya_ft.zip")
        print("saved to Drive: laya_ft.zip")
    else:
        try:
            from google.colab import files; files.download("/content/laya_ft.zip")
        except Exception as e:
            print("download skipped:", e)

---

## Where to take this further

* **More reward, less leakage:** everything proper-scoring lives in `laya.common.proper_reward` —
  tune `w_sph`/`w_rps`, or add a KL-to-reference penalty by keeping a frozen copy. The `td_lambda_targets`
  helper supports multi-turn trajectories if you build episode data.
* **High-cardinality labels:** raise `--max-options` in `curate.py` *together with*
  `agent.cfg["head_max_len"]` (e.g. 512) if you genuinely need >12-option questions; the README's
  Banking77 note explains the tradeoff.
* **Your own domain:** add a builder to `datasets/curate.py` (10 rows) pointing at any HF/Kaggle
  source, or hand-write JSONL in the exact schema — the QC cell enforces it.
* **Serve it:** [notebooks/laya_colab_openai_api.ipynb](https://github.com/tshewangrinzin/laya/blob/arena/01a0c378-laya/notebooks/laya_colab_openai_api.ipynb)
  exposes this checkpoint over the OpenAI-compatible API + Cloudflare tunnel.
* **Benchmarks:** the repo's `research/scripts/laya_benchmark_colab.ipynb` measures both
  checkpoints across 51 languages — a good before/after harness for your fine-tune.
* **Honest expectation:** one T4 and ~20 k curated cases will *not* reproduce 0.766 on
  typed-decisions by itself (the reference run trained on that benchmark's own train split); it
  should beat the base model's zero-shot behaviour on your workflows, with a far better ECE.